In [5]:
!uv add gitsource

Resolved 129 packages in 119ms
Checked 125 packages in 179ms


In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [7]:
len(files)

72

In [8]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [9]:
len(documents)

72

In [10]:
# from ingest import build_index
# lesson_index = build_index(documents)
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index


In [11]:
lesson_index = build_index(documents)

In [12]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = lesson_index.search(
    question,
    num_results=2
)

In [13]:

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [14]:
import json
print(json.dumps(search_results, indent=4))

[
    {
        "content": "# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don't know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it's\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver's seat, we have an agent. It's an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can cal

In [15]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [16]:
from hw1_rag_helper import RAGBase
assistant = RAGBase(lesson_index, openai_client)

In [17]:
assistant.rag(question)

('The agentic loop keeps calling the model by using a `while True` loop.\n\nEach iteration it:\n\n1. sends the full message history to the model,\n2. checks the response for any `function_call` items,\n3. runs those tool calls and appends the results to memory,\n4. repeats.\n\nIt stops when `has_function_calls` is `False`, meaning the model returned a final message with no more tool calls.',
 7036)

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
len(chunks)

295

In [21]:
chunk_index = build_index(chunks)

In [22]:
assistant = RAGBase(chunk_index, openai_client)

In [23]:
assistant.rag(question)

('It keeps calling the model inside a `while True` loop.\n\nEach turn, the code checks whether the model returned any `function_call` items:\n\n- If yes, it runs those tools, adds the results to `messages`, and loops again.\n- If no, it breaks out of the loop and stops.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, the loop continues until the model gives a final response with no more tool calls.',
 2221)

In [24]:
!uv add toyaikit

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

Resolved 129 packages in 1ms
Checked 125 packages in 3ms


In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5
    )

instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."


In [26]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

agent_tools = Tools()
agent_tools.add_tool(search)


In [27]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [28]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
